# 02 — Quantize one model: RTN-4 and GPTQ-4 (same grid), C4 or math calibration

Makes the two 4-bit versions of **one** Qwen2.5-Instruct model with exactly the settings of our 1.5B study:
- **RTN-4:** round every weight to the nearest point of its group's grid.
- **GPTQ-4:** same grid, rounding with error compensation; **act-order + static groups**
  (as in the authors' official code), C4 calibration: 128 × 2,048 tokens, seed 0, damping 1%.
- Both: 4 bits, asymmetric, group size 128, all linear layers of every block.

Then measures **perplexity** (WikiText2 and C4) of FP16, RTN and GPTQ as a sanity check.

**Change only `SIZE` in cell 2** ("0.5b", "1.5b", "3b").

**`CALIB = "math"`** (secondary question of the survey): makes only a GPTQ-4 model calibrated on
**GSM8K training problems with their solutions** (the test set is never used), with exactly the same
grid and settings as the C4 GPTQ (act-order + static groups, 128 × 2,048 tokens, seed 0, damping 1%).
RTN is skipped (it does not use calibration data). Output: `qwen2.5-<SIZE>-gptq4-g128-math-act/`,
`02_qwen2.5-<SIZE>_math_results.json`.

**Run:** GPU **T4**, Internet **On**, then **Save Version → Save & Run All (Commit)**.
Checkpointed: if it stops, attach this notebook's failed version as an Input and run again.

**Output (attach this notebook as an Input of notebook 01):**
`qwen2.5-<SIZE>-rtn4-g128/`, `qwen2.5-<SIZE>-gptq4-g128-c4-act/`, `02_qwen2.5-<SIZE>_results.json`

In [ ]:
# 1. Versions and GPU
import sys, math, torch, transformers, datasets
print("torch", torch.__version__, "| transformers", transformers.__version__,
      "| GPU", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE - turn on the GPU!")

In [ ]:
# 2. Settings  -- change only SIZE (and CALIB for the math-calibration run)
SIZE  = "1.5b"                     # "0.5b", "1.5b" or "3b"
CALIB = "c4"                       # "c4" = main study (RTN + GPTQ); "math" = GPTQ calibrated on GSM8K train only

MODEL_ID = {"0.5b": "Qwen/Qwen2.5-0.5B-Instruct", "1.5b": "Qwen/Qwen2.5-1.5B-Instruct",
            "3b": "Qwen/Qwen2.5-3B-Instruct", "7b": "Qwen/Qwen2.5-7B-Instruct"}[SIZE]
SHORT = f"qwen2.5-{SIZE}"
BITS, GROUP, NSAMPLES, SEQLEN, SEED, DAMP = 4, 128, 128, 2048, 0, 0.01   # same as the 1.5B study
OUT_DIR   = "/kaggle/working"
RTN_NAME  = f"{SHORT}-rtn{BITS}-g{GROUP}"
GPTQ_NAME = f"{SHORT}-gptq{BITS}-g{GROUP}-{CALIB}-act"
KEY       = "gptq4" if CALIB == "c4" else "gptq4math"     # name of this GPTQ model in all later notebooks
print(MODEL_ID, "->", RTN_NAME, "and", GPTQ_NAME)

In [ ]:
# 3. RTN and GPTQ code (same file as the 1.5B study)
# RTN and GPTQ in plain PyTorch, following the original GPTQ code
# (IST-DASLab/gptq: quant.py, gptq.py, opt.py). Both methods use the SAME
# Quantizer, so they share exactly the same grid definition.
import copy
import math
import torch
import torch.nn as nn


# ---------------------------------------------------------------------------
# 1. The grid (asymmetric min-max, as in Section 2.1 of the survey)
# ---------------------------------------------------------------------------
def quantize(x, scale, zero, maxq):
    """q = clip(round(x/s) + z, 0, 2^b - 1);  w_hat = s * (q - z)"""
    q = torch.clamp(torch.round(x / scale) + zero, 0, maxq)
    return scale * (q - zero)


class Quantizer:
    def __init__(self, bits, sym=False):
        self.maxq = 2 ** bits - 1
        self.sym = sym

    def find_params(self, w):
        """One scale and zero-point per row of w (w = one group of columns)."""
        xmin = torch.minimum(w.min(dim=1).values, torch.zeros(1, device=w.device))
        xmax = torch.maximum(w.max(dim=1).values, torch.zeros(1, device=w.device))
        if self.sym:
            xmax = torch.maximum(xmin.abs(), xmax)
            xmin = -xmax
        both_zero = (xmin == 0) & (xmax == 0)
        xmin[both_zero], xmax[both_zero] = -1, 1
        self.scale = ((xmax - xmin) / self.maxq).unsqueeze(1)
        if self.sym:
            self.zero = torch.full_like(self.scale, (self.maxq + 1) / 2)
        else:
            self.zero = torch.round(-xmin / self.scale.squeeze(1)).unsqueeze(1)

    def __call__(self, x):
        return quantize(x, self.scale, self.zero, self.maxq)


# ---------------------------------------------------------------------------
# 2. RTN: round each group independently, no data
# ---------------------------------------------------------------------------
@torch.no_grad()
def rtn_linear_(layer, bits, groupsize, sym=False):
    W = layer.weight.data.float()
    cols = W.shape[1]
    g = cols if groupsize == -1 else groupsize
    qz = Quantizer(bits, sym)
    Q = torch.empty_like(W)
    for c in range(0, cols, g):
        qz.find_params(W[:, c:c + g])
        Q[:, c:c + g] = qz(W[:, c:c + g])
    layer.weight.data = Q.to(layer.weight.dtype)


# ---------------------------------------------------------------------------
# 3. GPTQ for one linear layer (Algorithm 1 of the paper)
# ---------------------------------------------------------------------------
class GPTQ:
    def __init__(self, layer):
        self.layer = layer
        cols = layer.weight.shape[1]
        self.H = torch.zeros((cols, cols), device=layer.weight.device, dtype=torch.float32)
        self.nsamples = 0

    def add_batch(self, inp):
        """Accumulate H = 2 X X^T / n from the inputs this layer receives."""
        inp = inp.reshape(-1, inp.shape[-1]).t().float()        # (cols, tokens)
        self.H *= self.nsamples / (self.nsamples + 1)
        self.nsamples += 1
        inp = math.sqrt(2 / self.nsamples) * inp
        self.H += inp @ inp.t()

    @torch.no_grad()
    def quantize_(self, bits, groupsize=128, sym=False, percdamp=0.01, blocksize=128,
                  act_order=False, static_groups=False):
        """act_order: quantize columns with the largest H diagonal first ("desc_act").
        static_groups: each group's scale/zero-point is fixed from the ORIGINAL weights,
        so GPTQ uses exactly the same grid as RTN (only rounding decisions differ)."""
        W = self.layer.weight.data.clone().float()
        rows, cols = W.shape
        qz = Quantizer(bits, sym)
        if groupsize == -1:
            qz.find_params(W)
        H = self.H
        del self.H
        dead = torch.diag(H) == 0                    # inputs that are always 0
        H[dead, dead] = 1
        W[:, dead] = 0
        groups = None
        if groupsize != -1 and (static_groups or act_order):
            groups = []
            for c in range(0, cols, groupsize):
                g = Quantizer(bits, sym)
                g.find_params(W[:, c:c + groupsize])
                groups.append(g)
        perm = None
        if act_order:
            perm = torch.argsort(torch.diag(H), descending=True)
            W = W[:, perm]
            H = H[perm][:, perm]
            invperm = torch.argsort(perm)
        damp = percdamp * torch.mean(torch.diag(H))  # 1% dampening
        H += damp * torch.eye(cols, device=H.device)
        # Cholesky form of H^-1 (upper triangular), as in the paper
        H = torch.linalg.cholesky(H)
        H = torch.cholesky_inverse(H)
        Hinv = torch.linalg.cholesky(H, upper=True)

        Q = torch.zeros_like(W)
        losses = torch.zeros(rows, device=W.device)
        for i1 in range(0, cols, blocksize):          # lazy batch updates
            i2 = min(i1 + blocksize, cols)
            W1 = W[:, i1:i2].clone()
            Q1 = torch.zeros_like(W1)
            Err1 = torch.zeros_like(W1)
            Hinv1 = Hinv[i1:i2, i1:i2]
            for i in range(i2 - i1):                  # fixed column order
                if groups is not None:
                    col = int(perm[i1 + i]) if perm is not None else i1 + i
                    qz = groups[col // groupsize]
                elif groupsize != -1 and (i1 + i) % groupsize == 0:
                    qz.find_params(W[:, i1 + i:i1 + i + groupsize])
                w = W1[:, i]
                d = Hinv1[i, i]
                q = qz(w.unsqueeze(1)).flatten()
                Q1[:, i] = q
                losses += (w - q) ** 2 / d ** 2 / 2
                err = (w - q) / d
                W1[:, i:] -= err.unsqueeze(1) @ Hinv1[i, i:].unsqueeze(0)
                Err1[:, i] = err
            Q[:, i1:i2] = Q1
            W[:, i2:] -= Err1 @ Hinv[i1:i2, i2:]
        if perm is not None:
            Q = Q[:, invperm]
        self.layer.weight.data = Q.to(self.layer.weight.dtype)
        return losses.sum().item()


# ---------------------------------------------------------------------------
# 4. Whole-model helpers
# ---------------------------------------------------------------------------
def get_layers(model):
    """Transformer blocks for Qwen/Llama-style and OPT-style models."""
    inner = model.model
    if hasattr(inner, "layers"):
        return inner.layers
    if hasattr(inner, "decoder") and hasattr(inner.decoder, "layers"):
        return inner.decoder.layers
    raise ValueError("Unknown architecture")


def block_linears(block):
    return {n: m for n, m in block.named_modules() if isinstance(m, nn.Linear)}


@torch.no_grad()
def rtn_model_(model, bits=4, groupsize=128, sym=False):
    for block in get_layers(model):
        for m in block_linears(block).values():
            rtn_linear_(m, bits, groupsize, sym)


class _Catcher(nn.Module):
    """Stands in for block 0 to record its inputs, then stops the forward pass."""
    def __init__(self, m, store):
        super().__init__()
        self.m = m
        self.store = store

    def __getattr__(self, name):          # pass through attributes like attention_type
        try:
            return super().__getattr__(name)
        except AttributeError:
            return getattr(self.m, name)

    def forward(self, *args, **kwargs):
        hs = args[0] if args else kwargs.pop("hidden_states")
        self.store["inps"].append(hs)
        if self.store["kwargs"] is None:
            self.store["kwargs"] = kwargs
        raise ValueError("captured")


def _first(out):
    return out[0] if isinstance(out, (tuple, list)) else out


# Order used by "true sequential" GPTQ (Llama/Qwen-style blocks)
TRUE_SEQ = [["self_attn.q_proj", "self_attn.k_proj", "self_attn.v_proj"],
            ["self_attn.o_proj"], ["mlp.gate_proj", "mlp.up_proj"], ["mlp.down_proj"]]


@torch.no_grad()
def gptq_model_(model, calib, bits=4, groupsize=128, sym=False,
                percdamp=0.01, blocksize=128, verbose=True,
                act_order=False, static_groups=False, true_sequential=False):
    """Quantize block by block. Each block is calibrated on the outputs of the
    blocks before it that are ALREADY quantized (as in the GPTQ paper)."""
    layers = get_layers(model)
    dev = next(model.parameters()).device
    store = {"inps": [], "kwargs": None}
    layers[0] = _Catcher(layers[0], store)
    for x in calib:
        try:
            model(x.to(dev), use_cache=False)
        except ValueError:
            pass
    layers[0] = layers[0].m
    inps = torch.cat(store["inps"], dim=0)
    kw = store["kwargs"]
    outs = torch.zeros_like(inps)

    for li, block in enumerate(layers):
        lins = block_linears(block)
        names = list(lins)
        if true_sequential and all(n in lins for grp in TRUE_SEQ for n in grp) \
                and len(names) == sum(len(g) for g in TRUE_SEQ):
            steps = TRUE_SEQ     # quantize q/k/v, then o, then gate/up, then down
        else:
            steps = [names]      # all layers of the block at once (original GPTQ code)
        err = {}
        for step in steps:
            gptq = {n: GPTQ(lins[n]) for n in step}
            hooks = [lins[n].register_forward_hook(
                        lambda mod, inp, out, n=n: gptq[n].add_batch(inp[0].data))
                     for n in step]
            for j in range(inps.shape[0]):
                block(inps[j:j + 1], **kw)
            for h in hooks:
                h.remove()
            for n in step:
                err[n] = gptq[n].quantize_(bits, groupsize, sym, percdamp, blocksize,
                                           act_order, static_groups)
                del gptq[n]
        for j in range(inps.shape[0]):           # outputs of the QUANTIZED block
            outs[j] = _first(block(inps[j:j + 1], **kw))[0]
        inps, outs = outs, inps
        if verbose:
            print(f"  block {li:2d} done | total error {sum(err.values()):.1f}")
        torch.cuda.empty_cache() if torch.cuda.is_available() else None


# ---------------------------------------------------------------------------
# 5. Check: every (row, group) uses at most 2^bits distinct values
# ---------------------------------------------------------------------------
@torch.no_grad()
def check_grid(model, bits, groupsize):
    worst = 0
    for block in get_layers(model):
        for m in block_linears(block).values():
            W = m.weight.data.float()
            g = W.shape[1] if groupsize == -1 else groupsize
            for c in range(0, W.shape[1], g):
                grp = W[:, c:c + g]
                worst = max(worst, max(len(torch.unique(r)) for r in grp[:8]))
    return worst

In [ ]:
# 4. Data, perplexity, checkpointing
import os, glob, shutil, random, gc, json, time, re
import torch.nn.functional as F
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM

c4_train = load_dataset("allenai/c4", data_files={"train": "en/c4-train.00000-of-01024.json.gz"}, split="train")
c4_val = load_dataset("allenai/c4", data_files={"validation": "en/c4-validation.00000-of-00008.json.gz"},
                      split="validation")
wiki_text = "\n\n".join(load_dataset("Salesforce/wikitext", "wikitext-2-raw-v1", split="test")["text"])
tok = AutoTokenizer.from_pretrained(MODEL_ID)

def get_c4_calib(n=NSAMPLES, seqlen=SEQLEN, seed=SEED):     # identical to notebooks 02 / 02b
    random.seed(seed)
    out = []
    while len(out) < n:
        ids = tok(c4_train[random.randint(0, len(c4_train) - 1)]["text"], return_tensors="pt").input_ids
        if ids.shape[1] <= seqlen:
            continue
        i = random.randint(0, ids.shape[1] - seqlen - 1)
        out.append(ids[:, i:i + seqlen])
    return out

gsm_train = load_dataset("openai/gsm8k", "main", split="train")   # TRAIN split only (no test leakage)

def gsm8k_text(ex):                  # identical to notebook 02 (1.5B study)
    sol = re.sub(r"<<[^>]*>>", "", ex["answer"])                 # calculator notes removed
    sol = re.sub(r"####\s*(.+)$", r"The answer is \1.", sol.strip())
    return "Question: " + ex["question"] + "\nAnswer: " + sol

def get_math_calib(n=NSAMPLES, seqlen=SEQLEN, seed=SEED):
    # same shape as the C4 calibration (n pieces of seqlen tokens), but the text is GSM8K
    # training problems with their worked solutions, shuffled and joined into one stream
    order = list(range(len(gsm_train)))
    random.Random(seed).shuffle(order)
    ids = tok("\n\n".join(gsm8k_text(gsm_train[i]) for i in order), return_tensors="pt").input_ids
    rng = random.Random(seed)
    return [ids[:, i:i + seqlen] for i in (rng.randint(0, ids.shape[1] - seqlen - 1) for _ in range(n))]

WIKI_IDS = tok(wiki_text, return_tensors="pt").input_ids
C4_IDS = tok(" ".join(c4_val[:1100]["text"]), return_tensors="pt").input_ids[:, :256 * SEQLEN]

@torch.no_grad()
def ppl(model, ids, seqlen=SEQLEN):
    n = ids.shape[1] // seqlen
    nll = 0.0
    for i in range(n):
        x = ids[:, i * seqlen:(i + 1) * seqlen].to(model.device)
        logits = model(x, use_cache=False).logits[:, :-1, :].float()
        nll += F.cross_entropy(logits.reshape(-1, logits.shape[-1]), x[:, 1:].reshape(-1)).item() * seqlen
        del logits
    return math.exp(nll / (n * seqlen))

def load(path):
    return AutoModelForCausalLM.from_pretrained(path, dtype=torch.float16, device_map={"": 0}).eval()

def free():
    gc.collect(); torch.cuda.empty_cache()

def restore(name):
    dst = os.path.join(OUT_DIR, name)
    if os.path.exists(dst):
        return True
    hits = glob.glob(f"/kaggle/input/**/{name}", recursive=True)
    if not hits:
        return False
    (shutil.copytree if os.path.isdir(hits[0]) else shutil.copy)(hits[0], dst)
    print("restored", name)
    return True

RES = f"02_{SHORT}_results.json" if CALIB == "c4" else f"02_{SHORT}_math_results.json"
res = json.load(open(os.path.join(OUT_DIR, RES))) if restore(RES) else {}
def save():
    json.dump(res, open(os.path.join(OUT_DIR, RES), "w"), indent=1)
print("already done:", list(res))

In [ ]:
# 5. FP16 perplexity (reference)
if "fp16_c4" not in res:
    t0 = time.time()
    m = load(MODEL_ID)
    res["fp16_wiki"] = ppl(m, WIKI_IDS); res["fp16_c4"] = ppl(m, C4_IDS)
    del m; free(); save()
    print(f"({time.time() - t0:.0f} s)")
print(f"FP16:  WikiText2 {res['fp16_wiki']:.3f} | C4 {res['fp16_c4']:.3f}")

In [ ]:
# 6. RTN-4bit: quantize, check, save, perplexity  (skipped for CALIB = "math": RTN uses no data)
if CALIB != "c4":
    print("CALIB = math -> RTN skipped")
elif "rtn4_c4" not in res or not restore(RTN_NAME):
    t0 = time.time()
    m = load(MODEL_ID)
    rtn_model_(m, BITS, GROUP)
    print(f"max distinct values per group: {check_grid(m, BITS, GROUP)} (must be <= {2**BITS})")
    res["rtn4_quant_s"] = time.time() - t0
    m.save_pretrained(f"{OUT_DIR}/{RTN_NAME}"); tok.save_pretrained(f"{OUT_DIR}/{RTN_NAME}")
    res["rtn4_wiki"] = ppl(m, WIKI_IDS); res["rtn4_c4"] = ppl(m, C4_IDS)
    del m; free(); save()
if CALIB == "c4":
    print(f"RTN-4:  WikiText2 {res['rtn4_wiki']:.3f} | C4 {res['rtn4_c4']:.3f} | quantize {res['rtn4_quant_s']:.0f} s")

In [ ]:
# 7. GPTQ-4bit (act-order + static groups; C4 or math calibration): quantize, check, save, perplexity
if f"{KEY}_c4" not in res or not restore(GPTQ_NAME):
    t0 = time.time()
    calib = get_c4_calib() if CALIB == "c4" else get_math_calib()
    print("calibration:", CALIB, "| example:", repr(tok.decode(calib[0][0, :60])))
    m = load(MODEL_ID)
    gptq_model_(m, calib, BITS, GROUP, percdamp=DAMP, verbose=True,
                act_order=True, static_groups=True)
    print(f"max distinct values per group: {check_grid(m, BITS, GROUP)} (must be <= {2**BITS})")
    res[f"{KEY}_quant_s"] = time.time() - t0
    m.save_pretrained(f"{OUT_DIR}/{GPTQ_NAME}"); tok.save_pretrained(f"{OUT_DIR}/{GPTQ_NAME}")
    res[f"{KEY}_wiki"] = ppl(m, WIKI_IDS); res[f"{KEY}_c4"] = ppl(m, C4_IDS)
    del m; free(); save()
print(f"GPTQ-4 ({CALIB}): WikiText2 {res[f'{KEY}_wiki']:.3f} | C4 {res[f'{KEY}_c4']:.3f} | quantize {res[f'{KEY}_quant_s']/60:.1f} min")


In [ ]:
# 8. Summary
print(f"{SHORT}   {'WikiText2':>10s}{'C4':>9s}")
rows = [("FP16", "fp16"), ("RTN-4", "rtn4"), ("GPTQ-4", "gptq4")] if CALIB == "c4" else [("FP16", "fp16"), ("GPTQ-4 math", "gptq4math")]
for name, key in rows:
    print(f"{name:12s}{res[key + '_wiki']:10.3f}{res[key + '_c4']:9.3f}")
if CALIB == "c4":
    print("GPTQ beats RTN on both?", res["gptq4_wiki"] < res["rtn4_wiki"] and res["gptq4_c4"] < res["rtn4_c4"])
res["settings"] = dict(model=MODEL_ID, bits=BITS, group=GROUP, sym=False,
                       calib="C4 en shard 00000" if CALIB == "c4" else "GSM8K train (question + solution)",
                       nsamples=NSAMPLES, seqlen=SEQLEN, seed=SEED, damp=DAMP, act_order=True, static_groups=True)
save()
print("saved:", *([RTN_NAME] if CALIB == "c4" else []), GPTQ_NAME, RES)
